In [4]:
pip install seaborn


Note: you may need to restart the kernel to use updated packages.


In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [76]:
raw_df["Timestamp"] = pd.to_datetime(
    raw_df["Timestamp"],
    format="%d/%m/%Y %H:%M:%S",
    errors="coerce"
)

print("Invalid timestamps:", raw_df["Timestamp"].isna().sum())

Invalid timestamps: 0


In [77]:
raw_df = raw_df.sort_values("Timestamp").reset_index(drop=True)

In [78]:
print("Start:", raw_df["Timestamp"].min())
print("End:", raw_df["Timestamp"].max())

Start: 1970-01-05 03:01:17
End: 2018-02-14 12:59:59


In [79]:
numeric_cols = raw_df.select_dtypes(include=np.number).columns

raw_df[numeric_cols] = raw_df[numeric_cols].replace(
    [np.inf, -np.inf],
    np.nan
)

In [80]:
raw_df[numeric_cols] = raw_df[numeric_cols].fillna(
    raw_df[numeric_cols].median()
)

In [81]:
raw_df = raw_df.drop_duplicates().reset_index(drop=True)

In [82]:
NON_NEGATIVE_FEATURES = [
    "Flow Pkts/s",
    "Flow Duration",
    "Flow IAT Mean",
    "Flow IAT Std",
    "Flow IAT Max",
    "Flow IAT Min",
]

invalid_mask = (raw_df[NON_NEGATIVE_FEATURES] < 0).any(axis=1)

print("Invalid rows:", invalid_mask.sum())

raw_df = raw_df.loc[~invalid_mask].copy()
raw_df = raw_df.reset_index(drop=True)

Invalid rows: 5


In [83]:
print("Shape:", raw_df.shape)
print("Missing:", raw_df.isnull().sum().sum())
print(
    "Infinite:",
    np.isinf(
        raw_df.select_dtypes(include=np.number)
    ).sum().sum()
)
print(
    "Negative network values:",
    (raw_df[NON_NEGATIVE_FEATURES] < 0).sum().sum()
)
print("NaT timestamps:", raw_df["Timestamp"].isna().sum())

Shape: (822942, 80)
Missing: 0
Infinite: 0
Negative network values: 0
NaT timestamps: 0


In [84]:
PROCESSED_PATH = r"C:\code\\data\procesed\02-14-2018_cleaned.csv"

raw_df.to_csv(
    PROCESSED_PATH,
    index=False
)

print("Saved:", PROCESSED_PATH)

Saved: C:\code\\data\procesed\02-14-2018_cleaned.csv


In [85]:
test_df = pd.read_csv(PROCESSED_PATH)

test_df["Timestamp"] = pd.to_datetime(
    test_df["Timestamp"],
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce"
)

print("Shape:", test_df.shape)
print("NaT:", test_df["Timestamp"].isna().sum())
print(test_df["Timestamp"].head())

Shape: (822942, 80)
NaT: 0
0   2018-02-14 01:00:00
1   2018-02-14 01:00:00
2   2018-02-14 01:00:00
3   2018-02-14 01:00:00
4   2018-02-14 01:00:00
Name: Timestamp, dtype: datetime64[us]


In [87]:
DATA_PATH = r"C:\code\data\procesed\02-14-2018_cleaned.csv"

df = pd.read_csv(DATA_PATH)

df["Timestamp"] = pd.to_datetime(
    df["Timestamp"],
    format="%Y-%m-%d %H:%M:%S",
    errors="raise"
)

df = df.sort_values("Timestamp").reset_index(drop=True)

print(df.shape)
print(df["Timestamp"].min())
print(df["Timestamp"].max())

(822942, 80)
2018-02-14 01:00:00
2018-02-14 12:59:59


In [88]:
WINDOW = "5s"

df["time_window"] = df["Timestamp"].dt.floor(WINDOW)

print("Number of windows:", df["time_window"].nunique())
print(df[["Timestamp", "time_window"]].head(10))

Number of windows: 6509
            Timestamp         time_window
0 2018-02-14 01:00:00 2018-02-14 01:00:00
1 2018-02-14 01:00:00 2018-02-14 01:00:00
2 2018-02-14 01:00:00 2018-02-14 01:00:00
3 2018-02-14 01:00:00 2018-02-14 01:00:00
4 2018-02-14 01:00:00 2018-02-14 01:00:00
5 2018-02-14 01:00:00 2018-02-14 01:00:00
6 2018-02-14 01:00:00 2018-02-14 01:00:00
7 2018-02-14 01:00:00 2018-02-14 01:00:00
8 2018-02-14 01:00:00 2018-02-14 01:00:00
9 2018-02-14 01:00:00 2018-02-14 01:00:00


In [89]:
state_df = df.groupby("time_window").agg(
    {
        # Traffic volume
        "Tot Fwd Pkts": "sum",
        "Tot Bwd Pkts": "sum",
        "TotLen Fwd Pkts": "sum",
        "TotLen Bwd Pkts": "sum",

        # Traffic rates
        "Flow Byts/s": "mean",
        "Flow Pkts/s": "mean",

        # Flow behaviour
        "Flow Duration": "mean",
        "Fwd Pkt Len Mean": "mean",
        "Bwd Pkt Len Mean": "mean",
        "Fwd Pkt Len Std": "mean",
        "Bwd Pkt Len Std": "mean",

        # Timing
        "Flow IAT Mean": "mean",
        "Flow IAT Std": "mean",
        "Flow IAT Max": "max",
        "Flow IAT Min": "min",

        # TCP
        "SYN Flag Cnt": "sum",
        "ACK Flag Cnt": "sum",
        "RST Flag Cnt": "sum",
        "FIN Flag Cnt": "sum",
        "PSH Flag Cnt": "sum",

        # Packet size
        "Pkt Len Min": "mean",
        "Pkt Len Max": "max",
        "Pkt Len Mean": "mean",
        "Pkt Len Std": "mean",
        "Pkt Size Avg": "mean",

        # Network context
        "Dst Port": "nunique",
        "Protocol": "nunique",
    }
).reset_index()

In [93]:
df["IsAttack"] = (
    df["Label"].str.strip().str.lower() != "benign"
).astype(np.int8)

print(df["IsAttack"].value_counts())

IsAttack
0    666268
1    156674
Name: count, dtype: int64


In [95]:
print(df["Label"].value_counts())

Label
Benign            666268
SSH-Bruteforce    117322
FTP-BruteForce     39352
Name: count, dtype: int64


In [96]:
print("time_window" in df.columns)

True


In [97]:
attack_state = df.groupby("time_window").agg(
    total_flows=("Label", "size"),
    attack_flows=("IsAttack", "sum"),
    IsAttack=("IsAttack", "max")
).reset_index()

attack_state["attack_ratio"] = (
    attack_state["attack_flows"] /
    attack_state["total_flows"]
)

In [98]:
state_df = state_df.merge(
    attack_state,
    on="time_window",
    how="left"
)

In [99]:
print("Temporal states:", len(state_df))

print(
    state_df[
        [
            "time_window",
            "total_flows",
            "attack_flows",
            "attack_ratio",
            "IsAttack"
        ]
    ].head(20)
)

Temporal states: 6509
           time_window  total_flows  attack_flows  attack_ratio  IsAttack
0  2018-02-14 01:00:00           66             0           0.0         0
1  2018-02-14 01:00:05           63             0           0.0         0
2  2018-02-14 01:00:10          348             0           0.0         0
3  2018-02-14 01:00:15          200             0           0.0         0
4  2018-02-14 01:00:20          219             0           0.0         0
5  2018-02-14 01:00:25          343             0           0.0         0
6  2018-02-14 01:00:30           82             0           0.0         0
7  2018-02-14 01:00:35          319             0           0.0         0
8  2018-02-14 01:00:40          405             0           0.0         0
9  2018-02-14 01:00:45          256             0           0.0         0
10 2018-02-14 01:00:50          227             0           0.0         0
11 2018-02-14 01:00:55          191             0           0.0         0
12 2018-02-14 01

In [100]:
print(state_df["IsAttack"].value_counts())

IsAttack
0    4253
1    2256
Name: count, dtype: int64


In [101]:
print(state_df["attack_ratio"].describe())

count    6509.000000
mean        0.150917
std         0.256072
min         0.000000
25%         0.000000
50%         0.000000
75%         0.222222
max         0.915254
Name: attack_ratio, dtype: float64


In [102]:
print(df["Label"].value_counts())

Label
Benign            666268
SSH-Bruteforce    117322
FTP-BruteForce     39352
Name: count, dtype: int64


In [103]:
ATTACK_STAGE_MAP = {
    "Benign": "Benign",
    "SSH-Bruteforce": "Initial Access",
    "FTP-BruteForce": "Initial Access",
}

In [104]:
df["AttackStage"] = (
    df["Label"]
    .str.strip()
    .map(ATTACK_STAGE_MAP)
    .fillna("Unknown")
)

In [105]:
print(df["AttackStage"].value_counts())

AttackStage
Benign            666268
Initial Access    156674
Name: count, dtype: int64


In [106]:
def dominant_stage(series):
    counts = series.value_counts()

    if "Initial Access" in counts:
        return "Initial Access"

    return "Benign"

In [107]:
stage_state = df.groupby("time_window").agg(
    AttackStage=("AttackStage", dominant_stage)
).reset_index()

In [108]:
state_df = state_df.drop(
    columns=["AttackStage"],
    errors="ignore"
)

state_df = state_df.merge(
    stage_state,
    on="time_window",
    how="left"
)

In [109]:
print(
    state_df[
        ["time_window", "AttackStage"]
    ].head(20)
)

           time_window AttackStage
0  2018-02-14 01:00:00      Benign
1  2018-02-14 01:00:05      Benign
2  2018-02-14 01:00:10      Benign
3  2018-02-14 01:00:15      Benign
4  2018-02-14 01:00:20      Benign
5  2018-02-14 01:00:25      Benign
6  2018-02-14 01:00:30      Benign
7  2018-02-14 01:00:35      Benign
8  2018-02-14 01:00:40      Benign
9  2018-02-14 01:00:45      Benign
10 2018-02-14 01:00:50      Benign
11 2018-02-14 01:00:55      Benign
12 2018-02-14 01:01:00      Benign
13 2018-02-14 01:01:05      Benign
14 2018-02-14 01:01:10      Benign
15 2018-02-14 01:01:15      Benign
16 2018-02-14 01:01:20      Benign
17 2018-02-14 01:01:25      Benign
18 2018-02-14 01:01:30      Benign
19 2018-02-14 01:01:35      Benign


In [110]:
print(state_df["AttackStage"].value_counts())

AttackStage
Benign            4253
Initial Access    2256
Name: count, dtype: int64


In [111]:
print(
    state_df[
        [
            "time_window",
            "total_flows",
            "attack_flows",
            "attack_ratio",
            "IsAttack",
            "AttackStage"
        ]
    ].head(20)
)

           time_window  total_flows  attack_flows  attack_ratio  IsAttack  \
0  2018-02-14 01:00:00           66             0           0.0         0   
1  2018-02-14 01:00:05           63             0           0.0         0   
2  2018-02-14 01:00:10          348             0           0.0         0   
3  2018-02-14 01:00:15          200             0           0.0         0   
4  2018-02-14 01:00:20          219             0           0.0         0   
5  2018-02-14 01:00:25          343             0           0.0         0   
6  2018-02-14 01:00:30           82             0           0.0         0   
7  2018-02-14 01:00:35          319             0           0.0         0   
8  2018-02-14 01:00:40          405             0           0.0         0   
9  2018-02-14 01:00:45          256             0           0.0         0   
10 2018-02-14 01:00:50          227             0           0.0         0   
11 2018-02-14 01:00:55          191             0           0.0         0   

In [112]:
TEMPORAL_PATH = (
    r"C:\code\data\procesed"
    r"\02-14-2018_temporal_states.csv"
)

state_df.to_csv(
    TEMPORAL_PATH,
    index=False
)

print("Saved:", TEMPORAL_PATH)
print("Shape:", state_df.shape)

Saved: C:\code\data\procesed\02-14-2018_temporal_states.csv
Shape: (6509, 33)


In [115]:
import pandas as pd
import numpy as np

DATA_PATH = r"C:\code\data\procesed\02-14-2018_cleaned.csv"

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("NaT:", pd.to_datetime(
    df["Timestamp"],
    format="%d/%m/%Y %H:%M:%S",
    errors="coerce"
).isna().sum())

Shape: (822942, 80)
NaT: 822942
